# Redis caching and rate limiting

Implement cache-aside behavior and a fixed-window limiter, then reason about invalidation and distributed systems.

In [ ]:
# %pip install redis
import os, time
import redis

r = redis.Redis.from_url(os.environ.get('REDIS_URL', 'redis://localhost:6379/0'), decode_responses=True)
r.ping()

In [ ]:
def cache_key(project_id: int) -> str:
    return f'project:{project_id}:v1'

def get_cached_project(project_id: int, loader):
    key = cache_key(project_id)
    cached = r.get(key)
    if cached is not None:
        return cached, 'cache'
    value = loader(project_id)
    r.setex(key, 60, value)
    return value, 'database'

In [ ]:
def allow_request(subject: str, limit: int = 10, window: int = 60) -> bool:
    key = f'rate:{subject}:{int(time.time() // window)}'
    count = r.incr(key)
    if count == 1:
        r.expire(key, window)
    return count <= limit


## Exercises

1. Add cache invalidation after project updates.
2. Prevent cache stampedes for an expensive loader.
3. Compare fixed-window, sliding-window, and token-bucket approaches.
4. Make the limiter atomic under concurrency.
5. Decide which data must never be cached.
6. Measure cache hit ratio and stale-data behavior.
7. Simulate Redis outage and define graceful degradation.
8. Add per-user and per-IP limits with separate budgets.